### INITIALIZATION: 
IMPORT TOOLS, SET SEED, AND CREATE INDEPENDENT TABLES

In [1]:
# libraries
import pandas as pd
import numpy as np
import uuid
from datetime import datetime, timedelta

# seed
SEED = 42
np.random.seed(SEED)


print("Libraries loaded and seed set.")

Libraries loaded and seed set.


### Independent Entities

In [3]:
# Step 2: Independent Entities (Departments)
departments_data = [
    {"name": "Traffic & Transport", "daily_capacity": 50, "vulnerability_to_storm": 5.0, "base_rate": 20},
    {"name": "Public Works", "daily_capacity": 40, "vulnerability_to_storm": 8.0, "base_rate": 15},
    {"name": "Parks & Recreation", "daily_capacity": 10, "vulnerability_to_storm": 1.5, "base_rate": 2},
    {"name": "Animal Control", "daily_capacity": 15, "vulnerability_to_storm": 1.0, "base_rate": 5},
    {"name": "Sanitation", "daily_capacity": 60, "vulnerability_to_storm": 3.0, "base_rate": 45}
]

df_departments = pd.DataFrame(departments_data)

# Give each department a unique UUID
df_departments['department_id'] = [str(uuid.uuid4()) for _ in range(len(df_departments))]

print(df_departments.head())

                  name  ...                         department_id
0  Traffic & Transport  ...  7d1bf589-6ac8-4c4f-a174-860c07a596da
1         Public Works  ...  c7f876ca-990f-419d-932d-1c3dc2b3326b
2   Parks & Recreation  ...  86b4afe4-98d7-4ebe-b62e-e472fc98cf84
3       Animal Control  ...  0ac1b2a0-c490-43cb-8e03-60e9358890e4
4           Sanitation  ...  a7ab05ff-9ec6-4088-81eb-41ab8fa03761

[5 rows x 5 columns]


In [4]:
# setup 
NUM_CITIZENS = 10000
BASE_DATE = datetime(2026, 8, 1)

# citizen ids
citizen_ids = [str(uuid.uuid4()) for _ in range(NUM_CITIZENS)]

# dates
random_days_ago = np.random.randint(1, 365, size=NUM_CITIZENS)
join_dates = [
    (BASE_DATE - timedelta(days=int(days))).strftime("%Y-%m-%d") 
    for days in random_days_ago
]

# 5 DEMOGRAPHIC COLUMNS FOR CITIZEN PROFILING
ages = np.random.randint(18, 81, size=NUM_CITIZENS)
genders = np.random.choice(["Male", "Female", "Prefer not to say"], size=NUM_CITIZENS)
device_types = np.random.choice(["Android", "iOS", "Web"], size=NUM_CITIZENS)

makati_barangays = [
    "Poblacion", "Valenzuela", "Bel-Air", "San Lorenzo", "Urdaneta", "Magallanes", 
    "Dasmariñas", "Forbes Park", "Bangkal", "Pio del Pilar", "San Isidro", "Palanan", 
    "San Antonio", "La Paz", "Santa Cruz", "Singkamas", "Tejeros", "Kasilawan", 
    "Carmona", "Olympia", "Guadalupe Viejo", "Guadalupe Nuevo", "Pinagkaisahan", 
    "Pitogo", "South Cembo", "North Cembo", "Comembo", "East Rembo", "West Rembo", 
    "Pembo", "Rizal", "Post Proper Northside", "Post Proper Southside"
]
barangays = np.random.choice(makati_barangays, size=NUM_CITIZENS)
employment_statuses = np.random.choice(["Employed", "Unemployed", "Student", "Retired"], size=NUM_CITIZENS)

# clip L_civic
raw_scores = np.random.normal(loc=0.5, scale=0.3, size=NUM_CITIZENS)
L_civics = np.clip(raw_scores, a_min=0.1, a_max=2.0)

# creation of data frame
df_citizens = pd.DataFrame({
    "citizen_id": citizen_ids,
    "join_date": join_dates,
    "age": ages,
    "gender": genders,
    "device_type": device_types,
    "barangay": barangays,
    "employment_status": employment_statuses,
    "L_civic": L_civics,
})

print("=== DF_CITIZENS HEAD ===")
print(df_citizens.head())
print("\nL_civic Statistics:")
print(df_citizens["L_civic"].describe())


=== DF_CITIZENS HEAD ===
                             citizen_id   join_date  ...  employment_status   L_civic
0  9c0f35cb-0f3e-4e57-b7da-6c611aa89f48  2026-04-20  ...            Retired  0.658601
1  10fac9fb-4bc8-42ea-8cf6-6c25ba394a5f  2025-08-17  ...         Unemployed  0.409000
2  b28480a1-d531-438a-b8b3-adc88d5de256  2025-11-03  ...           Employed  0.518770
3  470c27b3-743d-4376-b9f4-d3243f59e23a  2026-04-16  ...           Employed  0.520398
4  e57b3586-387d-4f48-be87-1d6554b19d16  2026-05-21  ...            Student  0.378550

[5 rows x 8 columns]

L_civic Statistics:
count    10000.000000
mean         0.514104
std          0.276645
min          0.100000
25%          0.298502
50%          0.500355
75%          0.701882
max          1.572290
Name: L_civic, dtype: float64


### Step 3B: Timeline & Latent Weather Shock
Simulate a 30-day calendar starting from  (August 1, 2026) and inject a hidden omitted variable  representing a decaying typhoon shock.

In [6]:
# Step 3B: Timeline & Latent Weather Shock

# 1. 30 consecutive days starting from BASE_DATE using timedelta list comprehension
# create a list of all dates from the starting base date
timeline_dates = [
    (BASE_DATE + timedelta(days=i)).strftime("%Y-%m-%d")
    for i in range(30)
]

# 2. Vectorized initialization of latent storm shock (L_storm)
# intialize a 30 row list with 0.0
L_storm = np.zeros(30)

# 3. Inject decaying typhoon shock at specific indexes (Day 15, Day 16, Day 17)
# change the values to simulate a typoon situation
L_storm[14] = 1.0  # Index 14 (Day 15): Peak typhoon shock
L_storm[15] = 0.6  # Index 15 (Day 16): Receding floodwaters
L_storm[16] = 0.2  # Index 16 (Day 17): Residual shock

# Create timeline DataFrame
df_timeline = pd.DataFrame({
    "date": timeline_dates,
    "L_storm": L_storm
})

# Verify storm pulse injection
print("Timeline created. Storm pulse slice (index 13:18):")
print(df_timeline.iloc[13:18])

Timeline created. Storm pulse slice (index 13:18):
          date  L_storm
13  2026-08-14      0.0
14  2026-08-15      1.0
15  2026-08-16      0.6
16  2026-08-17      0.2
17  2026-08-18      0.0


### Step 3C: Timeline & Department Vulnerability Interaction Grid
Perform a Cartesian cross-join between `df_timeline` (30 days) and `df_departments` (5 departments) to create a 150-row simulation grid (`df_grid`).

In [8]:
# Step 3C: Timeline & Department Vulnerability Interaction Grid

# Perform Cartesian cross-join between timeline and departments
# merge the timeline with departments table, creating 150 rows
df_grid = df_timeline.merge(df_departments, how="cross")

# Verification
print(f"Simulation grid created with shape: {df_grid.shape}")
print("\nSample rows during storm peak (2026-08-15):")
print(df_grid[df_grid["date"] == "2026-08-15"][["date", "name", "L_storm", "vulnerability_to_storm", "base_rate"]])


Simulation grid created with shape: (150, 7)

Sample rows during storm peak (2026-08-15):
          date                 name  L_storm  vulnerability_to_storm  base_rate
70  2026-08-15  Traffic & Transport      1.0                     5.0         20
71  2026-08-15         Public Works      1.0                     8.0         15
72  2026-08-15   Parks & Recreation      1.0                     1.5          2
73  2026-08-15       Animal Control      1.0                     1.0          5
74  2026-08-15           Sanitation      1.0                     3.0         45


### Step 4: Core Ticket Generation Engine
Calculates expected daily ticket volume ($\lambda$) per department using $\lambda = \text{base\_rate} + (\text{base\_rate} \times L_{\text{storm}} \times \text{vulnerability\_to\_storm})$. Samples realized ticket counts using Poisson distribution (`np.random.poisson(lam)`) and assigns tickets to citizens weighted by their civic engagement score $L_{\text{civic}}$ (hyper-reporters).

In [10]:
# Step 4: Core Ticket Generation Engine
# Goal: Simulate how many complaints each department receives daily, and who reports them.

# 1. CALCULATE EXPECTED DAILY TICKETS (lambda / average daily rate)
# Formula: BaseRate + (BaseRate * L_storm * vulnerability_to_storm)
# Why: Sunny days (L_storm=0) keep base rates (e.g. 15). Storm days (L_storm=1.0) surge rates based on vulnerability.
df_grid["lambda"] = df_grid["base_rate"] + (
    df_grid["base_rate"] * df_grid["L_storm"] * df_grid["vulnerability_to_storm"]
)

# 2. GENERATE REALIZED DAILY TICKET COUNTS (Poisson Random Sampling)
# Why Poisson? Real city complaints fluctuate around an average. np.random.poisson(lam) generates realistic random counts.
df_grid["num_tickets"] = np.random.poisson(df_grid["lambda"])

# 3. CALCULATE CITIZEN REPORTING PROBABILITIES (Hyper-Reporter Weights)
# Why: Divide each citizen's L_civic score by the sum of all scores so total probability adds up to 100% (1.0).
citizen_weights = df_citizens["L_civic"].values / df_citizens["L_civic"].sum()

# 4. ASSIGN TICKETS TO CITIZENS (Weighted Random Choice / Raffle Drum)
# Why: np.random.choice picks citizens randomly, giving higher-scoring citizens a greater chance of being picked.
ticket_records = []
for idx, row in df_grid.iterrows():
    count = row["num_tickets"]
    if count > 0:
        assigned_citizens = np.random.choice(
            df_citizens["citizen_id"].values,
            size=count,
            p=citizen_weights
        )
        for c_id in assigned_citizens:
            ticket_records.append({
                "ticket_id": str(uuid.uuid4()),      # Unique 128-bit ID per ticket
                "created_at": row["date"],          # Creation date string (YYYY-MM-DD)
                "department_id": row["department_id"], # Department receiving the ticket
                "citizen_id": c_id                  # Assigned citizen ID
            })

# 5. CONSTRUCT FINAL TICKETS DATAFRAME
df_tickets = pd.DataFrame(ticket_records)

# Verification & Summaries (Pandas Pivot Table Grouping)
print(f"Total synthetic tickets generated: {len(df_tickets)}")
print("\nHead of df_tickets (First 5 rows):")
print(df_tickets.head())
print("\nDaily ticket counts during storm surge (2026-08-13 to 2026-08-18):")
daily_summary = df_grid.groupby("date")["num_tickets"].sum().reset_index()
print(daily_summary.iloc[12:18])


Total synthetic tickets generated: 3229

Head of df_tickets (First 5 rows):
                              ticket_id  ...                            citizen_id
0  d7978123-8bfa-422c-83c3-6f93697b5046  ...  5af7d443-ddf1-4864-a131-cc6aa8c9bfd0
1  59f5bb2c-23dd-46fb-b787-e25a3f8eae81  ...  6d6299f5-3503-4dad-b18d-88d8ada15bd2
2  271bf186-0d60-4100-8fc2-423f69c1cca5  ...  962adf3c-09af-436c-8f9c-5370b9f7a36a
3  c53f46a5-12a7-4c82-8633-49a3220c717c  ...  e5e8dc47-9ae8-497d-970f-9eda4b01ee57
4  4eff88c6-e864-4cfe-ac5f-6d725b7d6fcb  ...  b15896d6-c388-4503-a8e7-7eb8e777682e

[5 rows x 4 columns]

Daily ticket counts during storm surge (2026-08-13 to 2026-08-18):
          date  num_tickets
12  2026-08-13           84
13  2026-08-14           72
14  2026-08-15          466
15  2026-08-16          293
16  2026-08-17          157
17  2026-08-18           85


### Step 5: SLA Resolution Engine (Log-Normal Distribution)
Generates realistic resolution times in hours (`resolution_time_hours`) using a **Log-Normal distribution** (`np.random.lognormal`). Most tickets are resolved quickly (4–8 hours), but severe storm days trigger a heavy backlog shift that delays resolution times up to 70+ hours.

In [12]:
# Step 5: SLA Resolution Engine
# Goal: Calculate how long each ticket takes to resolve in hours.

# 1. MAP STORM SEVERITY (L_storm) TO TICKETS BY CREATION DATE
df_tickets = df_tickets.merge(df_timeline[["date", "L_storm"]], left_on="created_at", right_on="date", how="left")

# 2. GENERATE PRIORITY_LEVEL AND MAP PRIORITY_SHIFT MODIFIER
priority_levels = ["High", "Medium", "Low"]
priority_probs = [0.15, 0.60, 0.25]
df_tickets["priority_level"] = np.random.choice(priority_levels, p=priority_probs, size=len(df_tickets))

# Map priority_shift modifier: High = -0.5, Medium = 0.0, Low = +0.4
priority_map = {"High": -0.5, "Medium": 0.0, "Low": 0.4}
df_tickets["priority_shift"] = df_tickets["priority_level"].map(priority_map)

# 3. CALCULATE LOG-NORMAL PARAMETERS (STORM BACKLOG + PRIORITY SHIFT)
# meanlog = base_mean (1.5) + storm_backlog (L_storm * 1.2) + priority_shift
meanlog = 1.5 + (df_tickets["L_storm"] * 1.2) + df_tickets["priority_shift"]
sigmalog = 0.6

# 4. SAMPLE RESOLUTION TIMES AND ROUND TO 1 DECIMAL PLACE
df_tickets["resolution_time_hours"] = np.round(np.random.lognormal(meanlog, sigmalog), 1)

# Drop temporary shift modifier column
df_tickets = df_tickets.drop(columns=["priority_shift"])

# Verification: Summary statistics by priority level and storm level
print("=== RESOLUTION TIME BY PRIORITY LEVEL ===")
print(df_tickets.groupby("priority_level")["resolution_time_hours"].describe())


=== RESOLUTION TIME BY PRIORITY LEVEL ===
                 count       mean        std  min  25%  50%     75%   max
priority_level                                                           
High             462.0   4.641342   3.916332  0.5  2.1  3.4   5.875  35.4
Low              846.0  11.304374  10.183845  1.0  5.3  8.3  13.275  91.5
Medium          1921.0   7.526809   6.808553  0.7  3.3  5.4   9.100  67.8


### Step 6: Omitted Variable Removal & Final Dataset Export
To create a realistic machine learning benchmark, we **drop the hidden causal variables** (`L_storm` weather shock and `L_civic` engagement score) from the public tables, and export `departments.csv`, `citizens.csv`, and `tickets.csv` into the `data/` directory.

In [14]:
# Step 6: Omitted Variable Removal & Export
import os

# 1. APPEND FINAL POLISH COLUMNS (source_channel & ticket_status)
source_channels = ["Mobile App", "Hotline", "Web Portal"]
df_tickets["source_channel"] = np.random.choice(source_channels, size=len(df_tickets))
df_tickets["ticket_status"] = "Closed"

# 2. DROP LATENT OMITTED VARIABLES (L_storm from tickets, L_civic from citizens)
df_tickets_export = df_tickets.drop(columns=["L_storm", "date"])
df_citizens_export = df_citizens.drop(columns=["L_civic"])
df_departments_export = df_departments[["department_id", "name", "daily_capacity", "vulnerability_to_storm", "base_rate"]]

# 3. CREATE OUTPUT DATA DIRECTORY
os.makedirs("../data", exist_ok=True)

# 4. EXPORT CLEAN DATAFRAMES TO CSV FILES
df_departments_export.to_csv("../data/departments.csv", index=False)
df_citizens_export.to_csv("../data/citizens.csv", index=False)
df_tickets_export.to_csv("../data/tickets.csv", index=False)

# Verification & Total Column Quota Check
print("=== SYNTHETIC DATASET GENERATION COMPLETE ===")
print(f"Exported ../data/departments.csv -> Shape: {df_departments_export.shape} (Cols: {list(df_departments_export.columns)})")
print(f"Exported ../data/citizens.csv    -> Shape: {df_citizens_export.shape} (Cols: {list(df_citizens_export.columns)})")
print(f"Exported ../data/tickets.csv     -> Shape: {df_tickets_export.shape} (Cols: {list(df_tickets_export.columns)})")
total_cols = df_departments_export.shape[1] + df_citizens_export.shape[1] + df_tickets_export.shape[1]
print(f"\nTOTAL COLUMNS ACROSS ALL 3 TABLES: {total_cols} / 20+ quota met!")


=== SYNTHETIC DATASET GENERATION COMPLETE ===
Exported ../data/departments.csv -> Shape: (5, 5) (Cols: ['department_id', 'name', 'daily_capacity', 'vulnerability_to_storm', 'base_rate'])
Exported ../data/citizens.csv    -> Shape: (10000, 7) (Cols: ['citizen_id', 'join_date', 'age', 'gender', 'device_type', 'barangay', 'employment_status'])
Exported ../data/tickets.csv     -> Shape: (3229, 8) (Cols: ['ticket_id', 'created_at', 'department_id', 'citizen_id', 'priority_level', 'resolution_time_hours', 'source_channel', 'ticket_status'])

TOTAL COLUMNS ACROSS ALL 3 TABLES: 20 / 20+ quota met!
